In [3]:
import os

from google.cloud import bigquery

HTTP_PROXY = 'http://sia-lb.telekom.de:8080'
HTTPS_PROXY = 'http://sia-lb.telekom.de:8080'

os.environ['http_proxy'] = HTTP_PROXY
os.environ['https_proxy'] = HTTPS_PROXY

# --- 1. Initialize BigQuery client ---
PROJECT_ID = 'athlete-dashboard-467718'
DATASET_ID = 'strava_data_overview'
TABLE_ID = 'strava_data_test'
# client = bigquery.Client(project=PROJECT_ID)
client = bigquery.Client.from_service_account_json(
    'C:/Users/A200162055/Downloads/athlete-dashboard-467718-d5f31b294e6f.json',
    project='athlete-dashboard-467718',
)
print('BigQuery verbunden!')

full_table_id = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}'

# --- 4. Prüfen, ob Tabelle existiert ---
table = client.get_table(full_table_id)
print(f'ℹ️ Tabelle gefunden: {full_table_id}')
print(f'Spalten: {[schema.name for schema in table.schema]}')
print(f'Zeilen: {table.num_rows}')

# --- 5. Kleine Abfrage, um Daten zu testen ---
query = f'SELECT * FROM `{full_table_id}`'
print('🔹 Starte Abfrage:', query)

job = client.query(query)
df = job.result().to_dataframe()  # Ergebnisse in Pandas DataFrame

print('✅ Abfrage abgeschlossen, erste Zeilen:')
df

BigQuery verbunden!
ℹ️ Tabelle gefunden: athlete-dashboard-467718.strava_data_overview.strava_data_test
Spalten: ['Column', 'Dtype', 'Unique Count', 'Null Count', 'Missing %', 'Example Value', 'Description']
Geschätzte Zeilen: 58
🔹 Starte Abfrage: SELECT * FROM `athlete-dashboard-467718.strava_data_overview.strava_data_test`
✅ Abfrage abgeschlossen, erste Zeilen:


C:\Users\A200162055\Miniconda3\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Column,Dtype,Unique Count,Null Count,Missing %,Example Value,Description
0,trainer,bool,2,0,0.00,False,TODO: Add description for each column
1,commute,bool,1,0,0.00,False,TODO: Add description for each column
2,manual,bool,1,0,0.00,False,TODO: Add description for each column
3,private,bool,1,0,0.00,False,TODO: Add description for each column
4,flagged,bool,1,0,0.00,False,TODO: Add description for each column
5,has_heartrate,bool,2,0,0.00,True,TODO: Add description for each column
6,heartrate_opt_out,bool,1,0,0.00,False,TODO: Add description for each column
7,display_hide_heartrate_option,bool,2,0,0.00,True,TODO: Add description for each column
8,from_accepted_tag,bool,1,0,0.00,False,TODO: Add description for each column
9,has_kudoed,bool,1,0,0.00,False,TODO: Add description for each column


In [6]:
df.loc[df['Column'] == 'distance', 'Description'] = 'Distance of Activity in meters'

In [7]:
df

,Column,Dtype,Unique Count,Null Count,Missing %,Example Value,Description
0,trainer,bool,2,0,0.00,False,TODO: Add description for each column
1,commute,bool,1,0,0.00,False,TODO: Add description for each column
2,manual,bool,1,0,0.00,False,TODO: Add description for each column
3,private,bool,1,0,0.00,False,TODO: Add description for each column
4,flagged,bool,1,0,0.00,False,TODO: Add description for each column
5,has_heartrate,bool,2,0,0.00,True,TODO: Add description for each column
6,heartrate_opt_out,bool,1,0,0.00,False,TODO: Add description for each column
7,display_hide_heartrate_option,bool,2,0,0.00,True,TODO: Add description for each column
8,from_accepted_tag,bool,1,0,0.00,False,TODO: Add description for each column
9,has_kudoed,bool,1,0,0.00,False,TODO: Add description for each column


In [10]:
job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')

full_table_id = f'{PROJECT_ID}.{DATASET_ID}.activities_strava'

client.load_table_from_dataframe(df, full_table_id, job_config=job_config).result()

C:\Users\A200162055\Miniconda3\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=athlete-dashboard-467718, location=europe-west3, id=a257cc6e-2b75-4b3d-8fa6-8fed477fd856>